# PAR

This notebook is for exploring PAR values from all relevant deployments. It was sparked by noticing that many PAR values are less than zero, and wanting to explore the PAR data values across deployments

In [ ]:
from pathlib import Path

# import pandas as pd
import xarray as xr

from esdglider import gcp, paths, utils

home = Path.home()
deployments = [
    "unit_1024-20250224",
    "risso-20250414",
    "stenella-20250414",
    "risso-20260128",
    "stenella-20260128",
]

# gcp.gcs_mount_bucket(
#     paths.data_out_bucket_name, 
#     f"/home/user/mnt-gcs/{paths.data_out_bucket_name}", 
#     ro=True
# )

In [ ]:
for deployment_name in deployments:
    print(f"Processing deployment: {deployment_name} --------------------")
    glider_paths = paths.get_path_glider(
        deployment_name = deployment_name, 
        mode = "delayed", 
        home_path = home, 
    )

    par_var = "sci_bsipar_par"
    par_volts_var = "sci_bsipar_sensor_volts"

    try: 
        with xr.open_dataset(glider_paths["tsrawpath"]) as ds:
            sn, _ = utils.get_instrument_sn_date(ds, "instrument_par")
            print(f"PAR instrument serial number: {sn}")

            par_orig = ds.to_pandas()
            par_orig = par_orig[["sci_water_pressure", par_var, par_volts_var]]
            print(f"Number of total raw timeseries timestamps: {len(par_orig)}")

            par = par_orig[par_orig[par_var].notna()]
            print(f"Number of non-nan par values: {len(par)}")
            par.to_csv(f"/home/user/par/{deployment_name}_par.csv")

            print(f"Number of par values less than 0: {len(par[par[par_var] < 0])}")
            print(f"Number of par sensor voltage values less than 0: {len(par[par[par_volts_var] < 0])}")
            print(f"par min/max values: {par[par_var].min()} / {par[par_var].max()}")
            print(f"Number of par values greater than 3000 / 3500: {len(par[par[par_var] > 3000])} / {len(par[par[par_var] > 3500])}")

    except FileNotFoundError:
        print(f"File not found for deployment: {deployment_name}")

In [ ]:
ds

In [ ]:
# Exploration
deployment_name = "unit_1024-20250224"
glider_paths = paths.get_path_glider(
    deployment_name = deployment_name, 
    mode = "delayed", 
    home_path = home, 
)

ds_raw = xr.load_dataset(glider_paths["tsrawpath"])
display(ds_raw)
